# MFQATS: YOLOv12n Wood Surface Defect Detection (Training & Evaluation)

This notebook trains the **Detect Defects** model of the MFQATS inspection pipeline
(*Upload → 3D Reconstruction → Detect Defects → Result + Recommendation*) and produces the numbers and
figures for the capstone paper's results chapter.

| Item | Value |
|---|---|
| Model | YOLOv12n (Ultralytics file name `yolo12n.pt`), COCO-pretrained, fine-tuned |
| Dataset | Kaggle *Large Scale Image Dataset of Wood Surface Defects* (`nomihsa965`) |
| Split | 70% train / 20% validation / 10% test, fixed seed 42 |
| Training | 50 epochs, 640 px, batch 16, AMP, GPU 0, early stop after 20 epochs without improvement; flip / rotation / brightness augmentation |
| Reported metrics | Precision, Recall, F1-score, mAP@0.5, mAP@0.5:0.95 (overall and per class) on the **held-out test split** |
| Output | `best.pt` → copy to `src/api/weights/best.pt` in the MFQATS project |

**Where each paper test case is measured** (the last section fills in Pass/Fail for you):

| Paper | Test case | Notebook section |
|---|---|---|
| Table 20 – YOLO Model Training | Stable loss curves · augmented training · classes learned distinctly | 4, 5, 6, 9 |
| Table 21 – Testing Dataset | Held-out test set · real production-station photos | 6, 11 |
| Table 22 – Validation Dataset | No overfitting · best checkpoint by validation mAP · per-class validation | 5 |
| Table 23 – Precision/Recall Testing | P/R/F1/mAP per class · confidence threshold sweep · false positives on clean surfaces | 6, 7, 8 |
| Table 25 – Lighting Tests | Bright / dim lighting (simulated) | 9 |
| Section 3.8 – Performance | Detection within 3–5 s per image | 10 |

> **Scope note (for Chapter 1 limitations):** the Kaggle dataset has natural wood defects (knots, cracks, resin, marrow, quartzity …).
> Classes the paper also mentions (scratches, uneven finish, misaligned components) need MVCA's own annotated furniture photos.

**Before running**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. Click 🔑 **Secrets** (left sidebar), add `KAGGLE_API_TOKEN` with your Kaggle token, and turn on *Notebook access*.
   Never type the token into a cell.
3. `Runtime → Run all`. Allow Google Drive access when asked, because all results are saved there.

If Colab disconnects, just **Run all** again. Training resumes from the last saved epoch.

## 1. Setup: install dependencies, check GPU, mount Google Drive

In [ ]:
!pip install -q ultralytics kagglehub pyyaml
import torch, ultralytics
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)
assert torch.cuda.is_available(), 'GPU is OFF: Runtime -> Change runtime type -> T4 GPU, then Run all again'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/MFQATS_YOLO')   # everything that must survive a disconnect
RUNS_DIR = DRIVE_DIR / 'runs'
PAPER_DIR = DRIVE_DIR / 'paper_results'                  # tables + figures for the capstone paper
for d in (RUNS_DIR, PAPER_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Results will be saved to', DRIVE_DIR)

## 2. Download the dataset

In [ ]:
import os, kagglehub
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

DATASET_DIR = kagglehub.dataset_download('nomihsa965/large-scale-image-dataset-of-wood-surface-defects')
print('Downloaded to', DATASET_DIR)
!find "{DATASET_DIR}" -maxdepth 3 -type d | head -30

root = Path(DATASET_DIR)
for f in root.rglob('*'):   # files that may hold the class-name list
    if f.suffix.lower() in {'.yaml', '.yml', '.names'} or f.name.lower() in {'classes.txt', 'readme.txt', 'readme.md'}:
        print('===', f.relative_to(root)); print(f.read_text(errors='ignore')[:2000])
sample = next(f for f in root.rglob('*.txt') if f.name.lower() not in {'classes.txt', 'readme.txt'})
print('\nExample label file', sample.name, ':\n' + sample.read_text())

## 3. Build the YOLO dataset (train / val / test)
Class names are read from the dataset's own `data.yaml`/`classes.txt` so the label numbers keep their correct names.
Images are symlinked, not copied, to save disk space. The split is deterministic (seed 42), so a re-run after a
disconnect rebuilds the exact same split.

In [ ]:
import re, random, shutil, xml.etree.ElementTree as ET
from collections import Counter
from PIL import Image
import yaml

SRC = Path(DATASET_DIR)
OUT = Path('/content/wood_yolo')
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
SPLITS = {'train': 0.7, 'val': 0.2, 'test': 0.1}
SEED = 42

images = {p.stem: p for p in SRC.rglob('*') if p.suffix.lower() in IMG_EXT}
label_files = {}
for p in SRC.rglob('*'):
    if p.suffix.lower() in {'.txt', '.xml'} and p.stem in images and p.stem.lower() not in {'classes', 'readme'}:
        label_files.setdefault(p.stem, p)
print(f'{len(images)} images, {len(label_files)} annotation files')

yolo_names = None
for f in list(SRC.rglob('classes.txt')) + list(SRC.rglob('*.yaml')):
    if f.suffix == '.txt':
        yolo_names = [l.strip() for l in f.read_text().splitlines() if l.strip()]
    else:
        names = (yaml.safe_load(f.read_text()) or {}).get('names')
        yolo_names = [names[k] for k in sorted(names)] if isinstance(names, dict) else names
    if yolo_names:
        print('Class list from', f.name, '->', yolo_names)
        break
if not yolo_names:
    # The Kaggle YOLO labels ship without a class list. This order was verified: the per-id box counts rank exactly as
    # Kodytek et al. (2022) Table 1 (Live_Knot > Dead_Knot > Resin > Knot_with_crack > Crack > Marrow > Quartzity >
    # Knot_missing) and example crops of every id match their defect.
    yolo_names = ['Quartzity', 'Live_Knot', 'Marrow', 'Resin', 'Dead_Knot', 'Knot_with_crack', 'Knot_missing', 'Crack']
    print('Using verified class order ->', yolo_names)

def norm_name(n):
    return re.sub(r'\s+', '_', str(n).strip())

def parse(label_path, img_path):
    """Return [(class_name, cx, cy, w, h)] normalized 0-1. Handles YOLO txt, corner-format txt and Pascal VOC xml."""
    size = None
    def wh():
        nonlocal size
        if size is None:
            with Image.open(img_path) as im:
                size = im.size
        return size
    out = []
    if label_path.suffix.lower() == '.xml':
        for obj in ET.parse(label_path).getroot().iter('object'):
            bb = obj.find('bndbox')
            x1, y1, x2, y2 = (float(bb.find(k).text) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
            W, H = wh()
            out.append((norm_name(obj.find('name').text), (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H))
        return out
    for line in label_path.read_text(errors='ignore').splitlines():
        t = line.replace(',', ' ').split()
        if len(t) < 5:
            continue
        try:
            nums = [float(v) for v in t[-4:]]
        except ValueError:
            continue
        name = ' '.join(t[:-4])
        if re.fullmatch(r'\d+', name):                          # YOLO: class_id cx cy w h
            assert yolo_names, 'Numeric labels but no class list found - check the listing in step 2'
            out.append((norm_name(yolo_names[int(name)]), *nums))
            continue
        x1, y1, x2, y2 = nums                                    # ClassName x1 y1 x2 y2
        if max(nums) > 1.0:
            W, H = wh()
            x1, x2, y1, y2 = x1 / W, x2 / W, y1 / H, y2 / H
        out.append((norm_name(name), (x1 + x2) / 2, (y1 + y2) / 2, abs(x2 - x1), abs(y2 - y1)))
    return out

annotations = {stem: parse(lp, images[stem]) for stem, lp in label_files.items()}
counts = Counter(c for boxes in annotations.values() for c, *_ in boxes)
assert counts, 'No boxes parsed - inspect the folder listing in step 2'
CLASSES = yolo_names and [norm_name(n) for n in yolo_names] or sorted(counts)
CLASSES += [c for c in sorted(counts) if c not in CLASSES]

if OUT.exists():
    shutil.rmtree(OUT)
stems = sorted(annotations)
random.Random(SEED).shuffle(stems)
n = len(stems)
cut1, cut2 = int(n * SPLITS['train']), int(n * (SPLITS['train'] + SPLITS['val']))
split_of = {s: 'train' if i < cut1 else 'val' if i < cut2 else 'test' for i, s in enumerate(stems)}

clamp = lambda v: min(max(v, 0.0), 1.0)
for stem, split in split_of.items():
    (OUT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / split).mkdir(parents=True, exist_ok=True)
    img = images[stem]
    os.symlink(img, OUT / 'images' / split / img.name)
    lines = [f'{CLASSES.index(c)} {clamp(cx):.6f} {clamp(cy):.6f} {clamp(w):.6f} {clamp(h):.6f}'
             for c, cx, cy, w, h in annotations[stem] if w > 0 and h > 0]
    (OUT / 'labels' / split / f'{stem}.txt').write_text('\n'.join(lines))  # empty file = defect-free image

DATA_YAML = OUT / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(OUT), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
    'names': dict(enumerate(CLASSES)),
}, sort_keys=False))
print(DATA_YAML.read_text())

### 3.1 Check the classes match the MFQATS inference server
`src/api/yolo_server.py` assigns a severity and a rework recommendation to each class name.
A class missing from that list still works, but gets the generic "medium" severity.

In [ ]:
SERVER_CLASSES = {'crack', 'live_knot', 'dead_knot', 'knot_with_crack', 'knot_missing', 'marrow', 'resin',
                  'quartzity', 'blue_stain', 'overgrown', 'scratch', 'dent', 'unfinished_sanding', 'joint_misalignment'}
for c in CLASSES:
    key = re.sub(r'[\s-]+', '_', c.strip().lower())
    print(f'{c:20s}', 'OK (severity + recommendation mapped)' if key in SERVER_CLASSES else 'NOT MAPPED -> add it to DEFECT_INFO in yolo_server.py')

### 3.2 Dataset statistics (Table: dataset composition)

In [ ]:
import pandas as pd
rows = []
for split in SPLITS:
    split_stems = [s for s, sp in split_of.items() if sp == split]
    c = Counter(cl for s in split_stems for cl, *_ in annotations[s])
    rows.append({'Split': split, 'Images': len(split_stems),
                 'Defect-free images': sum(1 for s in split_stems if not annotations[s]),
                 **{cl: c.get(cl, 0) for cl in CLASSES}})
dataset_table = pd.DataFrame(rows).set_index('Split')
dataset_table.loc['Total'] = dataset_table.sum()
dataset_table.to_csv(PAPER_DIR / 'table_dataset_composition.csv')
dataset_table

### 3.3 Acceptance criteria (Pass/Fail thresholds)
The paper's test cases (Tables 20–25) say results must fall within an *"acceptable range defined by the team"*.
**Confirm these numbers with your team and adviser before running.** Every Pass/Fail below is computed against them.

In [ ]:
ACCEPTANCE = {
    'min_map50': 0.50,          # overall and per-class mAP@0.5 on test/validation
    'min_precision': 0.50,      # overall Precision on test
    'min_recall': 0.50,         # overall Recall on test
    'max_overfit_gap': 0.10,    # final val loss may be at most 10% above its lowest value
    'max_map_drop': 0.10,       # max mAP@0.5 drop under flips/rotation/lighting changes (absolute)
    'max_clean_fp_rate': 0.10,  # max share of defect-free images with a false detection
    'max_seconds': 5.0,         # Section 3.8: detection within 3-5 s per image
}

checks = {}  # (paper table, test case) -> result row; re-running a cell overwrites its own rows
def record(table, case, measured, criterion, passed):
    result = passed if isinstance(passed, str) else ('Pass' if passed else 'Fail')
    checks[(table, case)] = {'Measured': measured, 'Criterion': criterion, 'Result': result}
    print(f'[{result}] {table} | {case}: {measured}')

## 4. Train YOLOv12n (Table 20: YOLO Model Training)
Augmentation matches Table 20: **flipped** (horizontal + vertical), **rotated** (±10°) and **brightness-adjusted** (±40%) samples,
on top of Ultralytics' default mosaic/scale/translate.
Checkpoints are saved to Google Drive every epoch. If a previous run was interrupted, this cell **resumes** it.
Expect roughly 1–3 minutes per epoch on a T4 (check that `GPU_mem` is **not** `0G`).

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'wood_yolo12n'
RUN_DIR = RUNS_DIR / RUN_NAME
LAST = RUN_DIR / 'weights' / 'last.pt'
HYP = dict(epochs=50, imgsz=640, batch=16, amp=True, patience=20, seed=SEED,
           fliplr=0.5, flipud=0.5, degrees=10.0, hsv_v=0.4)

if LAST.exists():
    print('Resuming from', LAST)
    try:
        YOLO(str(LAST)).train(resume=True)
    except AssertionError as e:          # raised when that run already finished all epochs
        print('Nothing to resume:', e)
else:
    YOLO('yolo12n.pt').train(data=str(DATA_YAML), device=0, project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True, **HYP)

BEST = RUN_DIR / 'weights' / 'best.pt'
assert BEST.exists(), f'{BEST} not found - training did not finish'
best = YOLO(str(BEST))
print('Best weights:', BEST)

## 5. Training & validation analysis (Tables 20 and 22)
Loss curves (train vs validation), the overfitting check, which epoch was selected as best, and per-class validation results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

res = pd.read_csv(RUN_DIR / 'results.csv')
res.columns = res.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, loss in zip(axes, ['box_loss', 'cls_loss', 'dfl_loss']):
    ax.plot(res['epoch'], res[f'train/{loss}'], label='Training')
    ax.plot(res['epoch'], res[f'val/{loss}'], label='Validation')
    ax.set_title(loss.replace('_', ' ').title()); ax.set_xlabel('Epoch'); ax.legend()
fig.suptitle('YOLOv12n training vs validation loss')
fig.savefig(PAPER_DIR / 'fig_loss_curves_train_vs_val.png', dpi=150, bbox_inches='tight')
plt.show()

def metrics_table(m):
    """Overall + per-class Precision, Recall, F1, mAP from an Ultralytics val() result."""
    rows = {'All classes': (m.box.mp, m.box.mr, m.box.map50, m.box.map)}
    for i, c in enumerate(m.box.ap_class_index):
        rows[m.names[int(c)]] = m.box.class_result(i)
    t = pd.DataFrame(rows, index=['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95']).T.astype(float)
    t.insert(2, 'F1-score', 2 * t['Precision'] * t['Recall'] / (t['Precision'] + t['Recall']).replace(0, np.nan))
    return t.fillna(0).round(3)

first, last = res.iloc[0], res.iloc[-1]
record('Table 20', 'Model converges with stable loss curves',
       f"train box loss {first['train/box_loss']:.3f} -> {last['train/box_loss']:.3f}",
       'final training loss lower than first epoch', last['train/box_loss'] < first['train/box_loss'])

vl = res['val/box_loss']
gap = (vl.iloc[-1] - vl.min()) / vl.min()
record('Table 22', 'Validation loss decreases without overfitting',
       f'final val loss {gap:.1%} above its minimum (epoch {int(res.loc[vl.idxmin(), "epoch"])})',
       f"<= {ACCEPTANCE['max_overfit_gap']:.0%}", gap <= ACCEPTANCE['max_overfit_gap'])

map_col = 'metrics/mAP50-95(B)'
best_row = res.loc[res[map_col].idxmax()]
best_epoch = int(best_row['epoch'])
record('Table 22', 'Best checkpoint selected by highest validation mAP',
       f"best.pt = epoch {best_epoch} (val mAP@0.5:0.95 {best_row[map_col]:.3f}, mAP@0.5 {best_row['metrics/mAP50(B)']:.3f})",
       'best.pt is the highest-validation-mAP epoch', True)

val_m = best.val(data=str(DATA_YAML), split='val', imgsz=640, batch=16, device=0, plots=True,
                 project=str(RUNS_DIR), name='val_eval', exist_ok=True)
val_table = metrics_table(val_m)
val_table.to_csv(PAPER_DIR / 'table_validation_metrics_per_class.csv')
per_class = val_table.drop('All classes')
weak = per_class[per_class['mAP@0.5'] < ACCEPTANCE['min_map50']].index.tolist()
record('Table 22', 'Per-class validation accuracy within acceptable range',
       'all classes OK' if not weak else f'below threshold: {", ".join(weak)}',
       f"each class mAP@0.5 >= {ACCEPTANCE['min_map50']}", not weak)
val_table

## 6. Held-out TEST set evaluation (Tables 21 and 23)
**These are the main numbers to report in the paper.** The test images were never used for training or checkpoint selection.

In [ ]:
test = best.val(data=str(DATA_YAML), split='test', imgsz=640, batch=16, device=0, plots=True,
                project=str(RUNS_DIR), name='test_eval', exist_ok=True)
test_table = metrics_table(test)
test_table.to_csv(PAPER_DIR / 'table_test_metrics_per_class.csv')

allc = test_table.loc['All classes']
record('Table 21', 'Held-out test set: defects identified within acceptable accuracy',
       f"mAP@0.5 {allc['mAP@0.5']:.3f}", f"mAP@0.5 >= {ACCEPTANCE['min_map50']}", allc['mAP@0.5'] >= ACCEPTANCE['min_map50'])
record('Table 23', 'Precision, Recall, F1, mAP per defect class on test set',
       f"P {allc['Precision']:.3f} / R {allc['Recall']:.3f} / F1 {allc['F1-score']:.3f}",
       f"P >= {ACCEPTANCE['min_precision']} and R >= {ACCEPTANCE['min_recall']}",
       allc['Precision'] >= ACCEPTANCE['min_precision'] and allc['Recall'] >= ACCEPTANCE['min_recall'])
record('Table 20', 'Each defect class learned distinctly (no significant class confusion)',
       'see fig_confusion_matrix_test.png', 'inspect the off-diagonal cells of the confusion matrix', 'Manual')
test_table

## 7. Confidence threshold sweep (Table 23)
Precision, Recall and F1 on the test set at different confidence cutoffs. The cutoff with the highest F1 is the recommended
`CONF_THRESHOLD` for `src/api/yolo_server.py`.

In [ ]:
px = np.asarray(test.box.px)
P, R, F = (np.asarray(c).mean(0) for c in (test.box.p_curve, test.box.r_curve, test.box.f1_curve))
OPT_CONF = round(float(px[F.argmax()]), 2)

at = lambda curve, t: curve[np.abs(px - t).argmin()]
sweep = pd.DataFrame([{'Confidence threshold': t, 'Precision': at(P, t), 'Recall': at(R, t), 'F1-score': at(F, t)}
                      for t in [0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]]).round(3).set_index('Confidence threshold')
sweep.to_csv(PAPER_DIR / 'table_confidence_threshold_sweep.csv')

plt.figure(figsize=(7, 4))
plt.plot(px, P, label='Precision'); plt.plot(px, R, label='Recall'); plt.plot(px, F, label='F1-score')
plt.axvline(OPT_CONF, ls='--', c='gray', label=f'Optimal = {OPT_CONF}')
plt.xlabel('Confidence threshold'); plt.ylabel('Score'); plt.legend(); plt.title('Test set: metrics vs confidence threshold')
plt.savefig(PAPER_DIR / 'fig_confidence_threshold_sweep.png', dpi=150, bbox_inches='tight'); plt.show()

record('Table 23', 'Optimal confidence threshold balancing Precision and Recall',
       f'{OPT_CONF} (F1 {F.max():.3f})', 'threshold with the highest F1-score', True)
print(f'>>> Set CONF_THRESHOLD = {OPT_CONF} in src/api/yolo_server.py')
sweep

## 8. False positives on clean, defect-free surfaces (Table 23)

In [ ]:
clean = [OUT / 'images' / 'test' / images[s].name for s, sp in split_of.items() if sp == 'test' and not annotations[s]]
if clean:
    fp_imgs = sum(1 for r in best.predict([str(p) for p in clean], conf=OPT_CONF, imgsz=640, device=0, verbose=False, stream=True)
                  if len(r.boxes))
    fp_rate = fp_imgs / len(clean)
    record('Table 23', 'No or minimal false positives on defect-free surfaces',
           f'{fp_imgs}/{len(clean)} clean test images flagged ({fp_rate:.1%}) at conf {OPT_CONF}',
           f"<= {ACCEPTANCE['max_clean_fp_rate']:.0%}", fp_rate <= ACCEPTANCE['max_clean_fp_rate'])
else:
    record('Table 23', 'No or minimal false positives on defect-free surfaces',
           'dataset test split has no defect-free images', 'photograph clean MVCA surfaces and test manually', 'Manual')

## 9. Robustness: flipped, rotated and lighting variations (Tables 20 and 25)
The test set is re-evaluated after flipping, rotating, brightening and dimming every image (boxes transformed to match).
This simulates the lighting tests. Table 25's real station lighting tests still need photos taken at MVCA.

In [ ]:
from PIL import ImageEnhance, ImageOps
same = lambda cx, cy, w, h: (cx, cy, w, h)
VARIANTS = {
    'Original':               (lambda im: im, same),
    'Horizontal flip':        (ImageOps.mirror, lambda cx, cy, w, h: (1 - cx, cy, w, h)),
    'Vertical flip':          (ImageOps.flip, lambda cx, cy, w, h: (cx, 1 - cy, w, h)),
    'Rotated 180 deg':        (lambda im: im.rotate(180), lambda cx, cy, w, h: (1 - cx, 1 - cy, w, h)),
    'Bright lighting (+40%)': (lambda im: ImageEnhance.Brightness(im).enhance(1.4), same),
    'Dim lighting (-50%)':    (lambda im: ImageEnhance.Brightness(im).enhance(0.5), same),
}
test_stems = [s for s, sp in split_of.items() if sp == 'test']
rob_rows = []
for vname, (img_fn, box_fn) in VARIANTS.items():
    vdir = Path('/content/robustness') / re.sub(r'\W+', '_', vname)
    shutil.rmtree(vdir, ignore_errors=True)
    (vdir / 'images' / 'val').mkdir(parents=True); (vdir / 'labels' / 'val').mkdir(parents=True)
    for s in test_stems:
        with Image.open(images[s]) as im:
            img_fn(im.convert('RGB')).save(vdir / 'images' / 'val' / f'{s}.jpg', quality=95)
        lines = []
        for line in (OUT / 'labels' / 'test' / f'{s}.txt').read_text().splitlines():
            if line.strip():
                c, *b = line.split()
                lines.append(c + ' ' + ' '.join(f'{v:.6f}' for v in box_fn(*map(float, b))))
        (vdir / 'labels' / 'val' / f'{s}.txt').write_text('\n'.join(lines))
    y = vdir / 'data.yaml'
    y.write_text(yaml.safe_dump({'path': str(vdir), 'train': 'images/val', 'val': 'images/val', 'names': dict(enumerate(CLASSES))}))
    m = best.val(data=str(y), split='val', imgsz=640, batch=16, device=0, plots=False, verbose=False,
                 project=str(RUNS_DIR), name='robustness_eval', exist_ok=True)
    rob_rows.append({'Variation': vname, 'Precision': m.box.mp, 'Recall': m.box.mr, 'mAP@0.5': m.box.map50})

robust = pd.DataFrame(rob_rows).set_index('Variation')
robust['mAP@0.5 drop'] = robust.loc['Original', 'mAP@0.5'] - robust['mAP@0.5']
robust = robust.round(3)
robust.to_csv(PAPER_DIR / 'table_robustness_augmentation_lighting.csv')

geo = robust.loc[['Horizontal flip', 'Vertical flip', 'Rotated 180 deg'], 'mAP@0.5 drop'].max()
light = robust.loc[['Bright lighting (+40%)', 'Dim lighting (-50%)'], 'mAP@0.5 drop'].max()
record('Table 20', 'Generalizes across flipped / rotated variations', f'max mAP@0.5 drop {geo:.3f}',
       f"<= {ACCEPTANCE['max_map_drop']}", geo <= ACCEPTANCE['max_map_drop'])
record('Table 25', 'Detection under bright and dim lighting (simulated)', f'max mAP@0.5 drop {light:.3f}',
       f"<= {ACCEPTANCE['max_map_drop']}", light <= ACCEPTANCE['max_map_drop'])
robust

## 10. Processing time vs the 3–5 s per image requirement (Section 3.8)
Measured on GPU (Colab T4) and CPU, because the MFQATS inference server may run on a workstation without a GPU.
Time = preprocess + inference + postprocess, averaged over up to 50 test images after a warm-up run.

In [ ]:
speed_imgs = sorted((OUT / 'images' / 'test').iterdir())[:50]
speed_rows = []
for device in ['0', 'cpu']:
    m = YOLO(str(BEST))
    m.predict(str(speed_imgs[0]), imgsz=640, device=device, verbose=False)   # warm-up
    per_img_ms = []
    for p in speed_imgs:
        s = m.predict(str(p), imgsz=640, device=device, verbose=False)[0].speed
        per_img_ms.append(s['preprocess'] + s['inference'] + s['postprocess'])
    speed_rows.append({'Device': f'GPU ({torch.cuda.get_device_name(0)})' if device == '0' else 'CPU (Colab)',
                       'Images': len(per_img_ms),
                       'Avg time per image (s)': round(float(np.mean(per_img_ms)) / 1000, 3),
                       'Max time (s)': round(max(per_img_ms) / 1000, 3)})
speed_table = pd.DataFrame(speed_rows).set_index('Device')
speed_table.to_csv(PAPER_DIR / 'table_inference_speed.csv')

slowest = speed_table['Max time (s)'].max()
record('Section 3.8', 'Defect detection within 3-5 seconds per image',
       f"avg {speed_table['Avg time per image (s)'].max():.3f} s, max {slowest:.3f} s (slowest device)",
       f"<= {ACCEPTANCE['max_seconds']} s", slowest <= ACCEPTANCE['max_seconds'])
speed_table

## 11. (Optional) Photos from MVCA production stations (Table 21)
Upload real workshop photos to **Google Drive → `MyDrive/MFQATS_YOLO/shop_floor_images/`**, then run this cell.
Annotated results are saved to `paper_results/shop_floor/` to compare against the test-set results.

In [ ]:
SHOP_DIR = DRIVE_DIR / 'shop_floor_images'
SHOP_DIR.mkdir(exist_ok=True)
shop = sorted(p for p in SHOP_DIR.iterdir() if p.suffix.lower() in IMG_EXT)
if not shop:
    print('No shop-floor photos yet. Upload MVCA photos to', SHOP_DIR, 'and re-run this cell.')
    record('Table 21', 'Real production-station images consistent with test results', 'not run yet', 'visual comparison', 'Manual')
else:
    (PAPER_DIR / 'shop_floor').mkdir(exist_ok=True)
    n_det = 0
    for r in best.predict([str(p) for p in shop], conf=OPT_CONF, imgsz=640, device=0, verbose=False, stream=True):
        n_det += len(r.boxes)
        r.save(filename=str(PAPER_DIR / 'shop_floor' / f'{Path(r.path).stem}_pred.jpg'))
    record('Table 21', 'Real production-station images consistent with test results',
           f'{len(shop)} photos, {n_det} detections (see paper_results/shop_floor/)', 'visual comparison by inspectors', 'Manual')

## 12. Figures for the paper

In [ ]:
from IPython.display import Image as Show, display
figures = {
    'fig_training_curves.png': RUN_DIR / 'results.png',
    'fig_confusion_matrix_test.png': RUNS_DIR / 'test_eval' / 'confusion_matrix_normalized.png',
    'fig_precision_recall_test.png': RUNS_DIR / 'test_eval' / 'PR_curve.png',
    'fig_f1_confidence_test.png': RUNS_DIR / 'test_eval' / 'F1_curve.png',
    'fig_label_distribution.png': RUN_DIR / 'labels.jpg',
    'fig_augmented_training_batch.jpg': RUN_DIR / 'train_batch0.jpg',
}
for name, src in figures.items():
    if src.exists():
        shutil.copy(src, PAPER_DIR / name)
        print(name); display(Show(str(src), width=700))

preds = best.predict([str(p) for p in speed_imgs[:6]], imgsz=640, conf=OPT_CONF, device=0, verbose=False)
for i, r in enumerate(preds):
    out = PAPER_DIR / f'fig_sample_prediction_{i + 1}.jpg'
    r.save(filename=str(out))
    display(Show(str(out), width=700))

## 13. Test case results (fills the Result column of Tables 20–25) and export
Rows marked **Manual** need a human check (confusion matrix review, real workshop photos).
System-level tests (Table 16: live camera marking, capture without lag) are tested in the MFQATS app, not here.

In [ ]:
import json, datetime
results_df = pd.DataFrame([{'Paper table': t, 'Test case': c, **v} for (t, c), v in checks.items()]).sort_values('Paper table')
results_df.to_csv(PAPER_DIR / 'table_test_case_results.csv', index=False)
display(results_df.style.map(lambda v: 'color: #16a34a; font-weight: bold' if v == 'Pass'
                             else 'color: #dc2626; font-weight: bold' if v == 'Fail' else '', subset=['Result']))

summary = {
    'date': datetime.date.today().isoformat(),
    'model': 'YOLOv12n (Ultralytics yolo12n.pt, COCO-pretrained, fine-tuned)',
    'dataset': 'Kaggle nomihsa965/large-scale-image-dataset-of-wood-surface-defects',
    'classes': CLASSES,
    'split_images': {k: int(v) for k, v in dataset_table['Images'].drop('Total').items()},
    'hyperparameters': HYP,
    'best_epoch': best_epoch,
    'test_metrics': test_table.loc['All classes'].to_dict(),
    'optimal_confidence_threshold': OPT_CONF,
    'inference_speed_s': speed_table['Avg time per image (s)'].to_dict(),
    'acceptance_criteria': ACCEPTANCE,
    'test_case_results': results_df.to_dict('records'),
}
(PAPER_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=float))

shutil.copy(BEST, DRIVE_DIR / 'best.pt')
from google.colab import files
files.download(str(BEST))
print('Saved to Google Drive:', DRIVE_DIR / 'best.pt', '| paper tables & figures:', PAPER_DIR)
print(f'Next: put best.pt at src/api/weights/best.pt, set CONF_THRESHOLD = {OPT_CONF} in src/api/yolo_server.py,')
print('then run: uvicorn src.api.yolo_server:app --reload --port 8000')